> **Notebook-first lesson.** Examples are executable. Download-dependent examples are guarded so the notebook can still run offline.

## Mathematical Framework

Math companions for this lesson:

- [Math 03 · Probability & Bayes](../../math/03_probability_bayes.ipynb)
- [Math 05 · Information Theory](../../math/05_information_theory.ipynb)
- [Math 12 · Generative-Model Mathematics](../../math/12_generative_models_math.ipynb)

For this topic, explicitly state the **probabilistic/statistical model, objective, invariances, threshold/decision rule, and what assumptions connect the math to deployment data**.

# Lesson 46: Autoencoders and VAEs

## Autoencoder idea
An autoencoder learns to compress an input into a latent representation and reconstruct the original input.

Pipeline:

x -> encoder -> z -> decoder -> x_hat

The simplest objective is reconstruction loss.

## PyTorch skeleton


In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
import torch
from torch import nn

class Autoencoder(nn.Module):
    def __init__(self, d_in=784, d_latent=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, 256),
            nn.ReLU(),
            nn.Linear(256, d_latent),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_latent, 256),
            nn.ReLU(),
            nn.Linear(256, d_in),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z



## Latent space
The latent vector is a compressed learned representation. Inspect it using PCA or simple scatter plots.

## Variational autoencoders
A VAE learns a distribution over latent variables rather than one deterministic code.

Key ideas:
- mean and log variance
- reparameterization trick
- reconstruction term
- KL-divergence regularization

## Exercise
Train an autoencoder on MNIST/FashionMNIST. Compare reconstruction quality as latent dimension changes. Then inspect whether neighboring latent vectors produce similar decoded outputs.

## Signal connection
Autoencoders can learn compact representations of spectra or spectrograms and may be useful for denoising and anomaly detection.


## Runnable activity
This is a reduced-scale experiment for the core mechanism. Run it first, then extend it.

In [ ]:
import torch
from torch import nn
torch.manual_seed(0)
X=torch.randn(512,20)
ae=nn.Sequential(nn.Linear(20,8),nn.ReLU(),nn.Linear(8,3),nn.Linear(3,8),nn.ReLU(),nn.Linear(8,20))
opt=torch.optim.Adam(ae.parameters(),lr=.01)
for _ in range(150):
    opt.zero_grad(); rec=ae(X); loss=((rec-X)**2).mean(); loss.backward(); opt.step()
print("autoencoder MSE",float(loss))
mu=torch.tensor([[1.,-1.]]); logvar=torch.tensor([[0.,np.log(4.)]]) if False else torch.tensor([[0.,1.3862944]])
eps=torch.zeros_like(mu)
z=mu+torch.exp(.5*logvar)*eps
print("reparameterized z with epsilon=0:",z)

## Explanation checkpoint
Explain the mechanism, the scale gap between this activity and production/research systems, and one experiment you would run next.